# Comparison: my pipeline vs. collaborator's final objects

**Goal:** isolate exactly which cells the collaborator removed between my post-QC
objects (~68k mouse / ~70k human) and his *final* published objects
(57,242 mouse / ... human), then **fingerprint** the removed cells by mechanism
so I know *who* is being removed and *why*:

- **unplaced** (no spatial coords) — dropped by something other than spatial QC
- **spatial-manual** — falls in the hardcoded exclusion boxes (2_dpi y>10000, 1_dpi_late x<1000)
- **spatial-kNN** — has coords, high kNN spatial-outlier distance
- **other** (lineage reintegration / cluster exclusions) — the residual

The join key is **(sample, cell_barcode)** — barcodes are only unique within a sample.
My concat already stripped the -1 suffix into `cell_barcode`; I must confirm his
barcodes match (stripped vs suffixed) or the join silently finds nothing.


In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd

# ---- paths ----
SPECIES = "mouse"   # flip to "human" and re-run

MINE = {
    "mouse": "data/interim/latent_mouse.h5ad",   # post-integration (or qc_passed_mouse.h5ad pre-integration)
    "human": "data/interim/latent_human.h5ad",
}
# Collaborator's FINAL objects (post-everything: QC + lineage + cluster exclusions)
THEIRS = {
    "mouse": "/n/eddy_lab/Lab/mckinley/data/2025_04_15_AmhrGsD_Final.h5ad",
    "human": "FILL_IN_HUMAN_FINAL_PATH.h5ad",     # <-- set this
}

mine_path, theirs_path = MINE[SPECIES], THEIRS[SPECIES]
print("mine  :", mine_path)
print("theirs:", theirs_path)


In [ ]:
# Load both (backed='r' = read-only, fast, low memory — we only need obs)
mine = sc.read_h5ad(mine_path, backed="r")
theirs = sc.read_h5ad(theirs_path, backed="r")

print(f"MINE  : {mine.n_obs:,} cells")
print(f"THEIRS: {theirs.n_obs:,} cells")
print(f"diff  : {mine.n_obs - theirs.n_obs:,} cells removed by collaborator downstream")
print()
print("MINE obs cols:", list(mine.obs.columns))
print("THEIRS obs cols:", list(theirs.obs.columns))


## Step 0 — confirm the barcode formats match before joining

If mine are stripped (`AGCT...`) and his are suffixed (`AGCT...-1`), the join finds
**nothing** and every cell looks "removed". Check a few from each side.


In [ ]:
# my barcodes (concat stripped the -1 into cell_barcode)
mine_bc_col = "cell_barcode" if "cell_barcode" in mine.obs.columns else None
print("MINE   cell_barcode samples:", list(mine.obs[mine_bc_col].iloc[:3]) if mine_bc_col else "(no cell_barcode col!)")
print("MINE   index samples:       ", list(mine.obs_names[:3]))

# his barcodes — find the column (could be cell_barcode, or in the index)
theirs_bc_col = "cell_barcode" if "cell_barcode" in theirs.obs.columns else None
print("THEIRS cell_barcode samples:", list(theirs.obs[theirs_bc_col].iloc[:3]) if theirs_bc_col else "(no cell_barcode col)")
print("THEIRS index samples:       ", list(theirs.obs_names[:3]))

# also confirm the sample column name on his side
print()
print("THEIRS sample col present:", "sample" in theirs.obs.columns)
print("THEIRS timepoint col present:", "timepoint" in theirs.obs.columns)


In [ ]:
# Build a clean (sample, barcode) key on BOTH sides.
# Strip any trailing -<digits> so both are suffix-free, then join.

def clean_bc(series):
    return series.astype(str).str.replace(r"-\\d+$", "", regex=True)

# --- mine ---
m = mine.obs.copy()
if mine_bc_col:
    m["bc"] = clean_bc(m[mine_bc_col])
else:
    m["bc"] = clean_bc(pd.Series(mine.obs_names, index=mine.obs.index))
m["key"] = m["sample"].astype(str) + "||" + m["bc"]

# --- theirs ---
t = theirs.obs.copy()
if theirs_bc_col:
    t["bc"] = clean_bc(t[theirs_bc_col])
else:
    t["bc"] = clean_bc(pd.Series(theirs.obs_names, index=theirs.obs.index))
# his sample column — adjust if named differently
his_sample_col = "sample" if "sample" in t.columns else None
assert his_sample_col, "collaborator object has no 'sample' column — set his_sample_col manually"
t["key"] = t[his_sample_col].astype(str) + "||" + t["bc"]

mine_keys = set(m["key"])
theirs_keys = set(t["key"])
overlap = mine_keys & theirs_keys

print(f"mine keys   : {len(mine_keys):,}")
print(f"theirs keys : {len(theirs_keys):,}")
print(f"overlap     : {len(overlap):,}")
print()
if len(overlap) == 0:
    print("!!! ZERO overlap — barcode or sample formats don't match. Inspect the samples above.")
elif len(overlap) < 0.5 * len(theirs_keys):
    print(f"!!! LOW overlap ({100*len(overlap)/len(theirs_keys):.0f}% of his cells) — partial format mismatch?")
else:
    print(f"OK: {100*len(overlap)/len(theirs_keys):.1f}% of his cells matched in mine")


## Step 1 — the removed set

Cells in **mine** but not in **his final** = everything he dropped downstream
(spatial QC + lineage reintegration + cluster exclusions, all lumped).


In [ ]:
removed_keys = mine_keys - theirs_keys
m_removed = m[m["key"].isin(removed_keys)].copy()

print(f"removed: {len(m_removed):,} cells ({100*len(m_removed)/len(m):.1f}% of mine)")
print()
print("removed by timepoint:")
print(m_removed["timepoint"].value_counts())


## Step 2 — fingerprint the removed cells by mechanism

Partition the removed set: unplaced / spatial-manual-box / (coords-having residual).
The coords-having residual is then split into kNN-outlier vs other by computing the
per-sample kNN spatial distance (the collaborator's `filter_spatially` logic).


In [ ]:
# placement: does the cell have spatial coords?
has_coords = m_removed["x_um_dbscan"].notna() & m_removed["y_um_dbscan"].notna()

# manual exclusion boxes (collaborator's hardcoded cuts)
in_manual_box = (
    ((m_removed["timepoint"] == "2_dpi") & (m_removed["y_um_dbscan"] > 10000))
    | ((m_removed["timepoint"] == "1_dpi_late") & (m_removed["x_um_dbscan"] < 1000))
)

m_removed["fp_unplaced"] = ~has_coords
m_removed["fp_manual_box"] = has_coords & in_manual_box
m_removed["fp_coords_other"] = has_coords & ~in_manual_box  # kNN-outlier OR lineage/cluster

print("removed-cell fingerprint:")
print(f"  unplaced (no coords)     : {int(m_removed['fp_unplaced'].sum()):,}")
print(f"  in manual exclusion box  : {int(m_removed['fp_manual_box'].sum()):,}")
print(f"  has coords, other reason : {int(m_removed['fp_coords_other'].sum()):,}")
print()
print("By timepoint x mechanism:")
print(pd.crosstab(m_removed["timepoint"],
                  m_removed[["fp_unplaced","fp_manual_box","fp_coords_other"]].idxmax(axis=1)))


## Step 3 — kNN spatial-outlier check on the coords-having removed cells

For removed cells that HAD coords and aren't in a manual box, are they spatial
kNN outliers (far from neighbors)? Compute per-sample kNN mean distance on the
FULL placed set, then see whether removed cells sit in the high tail — that
distinguishes "spatially bad" from "removed for a lineage/cluster reason".


In [ ]:
from sklearn.neighbors import NearestNeighbors

K = 5            # collaborator used k=5 (later k=10); matches filter_spatially
KEEP_PCT = 0.95  # top 5% by mean kNN distance flagged as outliers

# work on ALL my placed cells (need the full spatial context per sample)
m_placed = m[m["x_um_dbscan"].notna() & m["y_um_dbscan"].notna()].copy()

knn_dist = pd.Series(np.nan, index=m_placed.index)
for s, sub in m_placed.groupby("sample"):
    X = sub[["x_um_dbscan", "y_um_dbscan"]].to_numpy(float)
    if len(sub) <= K + 1:
        continue
    nn = NearestNeighbors(n_neighbors=K + 1).fit(X)
    d, _ = nn.kneighbors(X)
    knn_dist.loc[sub.index] = d[:, 1:].mean(axis=1)

m_placed["knn_mean_dist"] = knn_dist
# per-sample 95th-percentile threshold
thr = m_placed.groupby("sample")["knn_mean_dist"].transform(lambda x: x.quantile(KEEP_PCT))
m_placed["is_knn_outlier"] = m_placed["knn_mean_dist"] > thr

# of the removed coords-having cells, how many are kNN outliers?
removed_placed_keys = set(m_removed.loc[m_removed["fp_coords_other"], "key"])
mp_removed = m_placed[m_placed["key"].isin(removed_placed_keys)]
n_knn = int(mp_removed["is_knn_outlier"].sum())
print(f"coords-having removed cells (non-manual): {len(removed_placed_keys):,}")
print(f"  of those, kNN spatial outliers : {n_knn:,}")
print(f"  remaining (lineage/cluster?)   : {len(removed_placed_keys) - n_knn:,}")


## Step 4 — summary table

The decomposition of the ~11k removed cells. The "lineage/cluster residual" is
what's left after spatial mechanisms — those are the cells removed by the
integration/lineage-separation steps I haven't reproduced yet (the 518/692
reintegration cells + annotation cluster exclusions).


In [ ]:
n_total = len(m_removed)
n_unplaced = int(m_removed["fp_unplaced"].sum())
n_manual = int(m_removed["fp_manual_box"].sum())
n_coords_other = int(m_removed["fp_coords_other"].sum())
# split coords_other into knn vs residual using Step 3
n_knn_out = n_knn
n_residual = n_coords_other - n_knn_out

summary = pd.DataFrame({
    "mechanism": ["unplaced (no coords)", "manual exclusion box",
                  "kNN spatial outlier", "lineage/cluster residual"],
    "n_cells": [n_unplaced, n_manual, n_knn_out, n_residual],
})
summary["pct_of_removed"] = (100 * summary["n_cells"] / n_total).round(1)
print(f"SPECIES = {SPECIES}")
print(f"total removed by collaborator: {n_total:,}")
print()
print(summary.to_string(index=False))
